# Лабораторная 2. Формирование отчётов в Apache Spark

## Задание

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.


## 1. Установка PySpark


In [ ]:
!pip install -q pyspark==3.5.1

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
BASE_PATH = "/content/drive/MyDrive/BD_LR"

POSTS_PATH = f"{BASE_PATH}/posts_sample.xml"
LANGUAGES_PATH = f"{BASE_PATH}/programming-languages.csv"
OUTPUT_PATH = f"{BASE_PATH}/top_languages_2010_2020.parquet"

print("POSTS_PATH =", POSTS_PATH)
print("LANGUAGES_PATH =", LANGUAGES_PATH)
print("OUTPUT_PATH =", OUTPUT_PATH)


POSTS_PATH = /content/drive/MyDrive/BD_LR/posts_sample.xml
LANGUAGES_PATH = /content/drive/MyDrive/BD_LR/programming-languages.csv
OUTPUT_PATH = /content/drive/MyDrive/BD_LR/top_languages_2010_2020.parquet


## 2. Импорты и создание SparkSession


In [22]:
import re
from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder
    .appName("Top programming languages 2010-2020")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.18.0")
    .getOrCreate()
)

spark


## 3. Функция нормализации названий языков

Она приводит названия из CSV к виду тегов Stack Overflow.


In [23]:
def normalize_language_to_tag(name: str):
    if name is None:
        return None

    s = name.strip().lower()
    if not s:
        return None

    # убираем содержимое в скобках
    s = re.sub(r"\s*\(.*?\)\s*", " ", s)

    # убираем хвосты вида " - ISO/IEC ..." или " – ISO/IEC ..."
    s = re.sub(r"\s*[–—-]\s*iso.*$", "", s)

    # убираем хвосты после запятой
    s = re.sub(r"\s*,.*$", "", s)

    # нормализуем пробелы
    s = re.sub(r"\s+", " ", s).strip()

    aliases = {
        "visual basic .net": "vb.net",
        "visual basic": "visual-basic",
        "objective c": "objective-c",
        "common lisp": "common-lisp",
        "assembly language": "assembly",
        "emacs lisp": "elisp",
        "wolfram language": "mathematica",
        "go!": "go",
    }

    if s in aliases:
        return aliases[s]

    return s.replace(" ", "-")


## 4. Чтение XML с постами


In [24]:
posts = (
    spark.read
    .format("xml")
    .option("rowTag", "row")
    .load(POSTS_PATH)
    .select(
        F.col("_Id").alias("post_id"),
        F.col("_PostTypeId").cast("int").alias("post_type_id"),
        F.col("_CreationDate").alias("creation_date"),
        F.col("_Tags").alias("tags")
    )
)

posts.show(10, truncate=False)
posts.printSchema()


+-------+------------+-----------------------+------------------------------------------------------+
|post_id|post_type_id|creation_date          |tags                                                  |
+-------+------------+-----------------------+------------------------------------------------------+
|4      |1           |2008-07-31 21:42:52.667|<c#><floating-point><type-conversion><double><decimal>|
|6      |1           |2008-07-31 22:08:08.62 |<html><css><internet-explorer-7>                      |
|7      |2           |2008-07-31 22:17:57.883|NULL                                                  |
|9      |1           |2008-07-31 23:40:59.743|<c#><.net><datetime>                                  |
|11     |1           |2008-07-31 23:55:37.967|<c#><datetime><time><datediff><relative-time-span>    |
|12     |2           |2008-07-31 23:56:41.303|NULL                                                  |
|13     |1           |2008-08-01 00:42:38.903|<html><browser><timezone><user-agent

## 5. Чтение CSV со списком языков


In [25]:
languages_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(LANGUAGES_PATH)
)

languages_raw.show(10, truncate=False)
print(languages_raw.columns)


+----------+---------------------------------------------------------+
|name      |wikipedia_url                                            |
+----------+---------------------------------------------------------+
|A# .NET   |https://en.wikipedia.org/wiki/A_Sharp_(.NET)             |
|A# (Axiom)|https://en.wikipedia.org/wiki/A_Sharp_(Axiom)            |
|A-0 System|https://en.wikipedia.org/wiki/A-0_System                 |
|A+        |https://en.wikipedia.org/wiki/A%2B_(programming_language)|
|A++       |https://en.wikipedia.org/wiki/A%2B%2B                    |
|ABAP      |https://en.wikipedia.org/wiki/ABAP                       |
|ABC       |https://en.wikipedia.org/wiki/ABC_(programming_language) |
|ABC ALGOL |https://en.wikipedia.org/wiki/ABC_ALGOL                  |
|ABSET     |https://en.wikipedia.org/wiki/ABSET                      |
|ABSYS     |https://en.wikipedia.org/wiki/ABSYS                      |
+----------+---------------------------------------------------------+
only s

## 6. Подготовка справочника языков


In [26]:
possible_cols = [
    c for c in languages_raw.columns
    if c.lower() in {"name", "language", "programming_language"}
]

language_col = possible_cols[0] if possible_cols else languages_raw.columns[0]

language_names = [
    row[0]
    for row in languages_raw.select(F.col(language_col)).distinct().collect()
    if row[0] is not None
]

tag_to_language = {}

for lang in language_names:
    tag = normalize_language_to_tag(lang)
    if tag and tag not in tag_to_language:
        tag_to_language[tag] = lang.strip()

languages_df = spark.createDataFrame(
    [(tag, display) for tag, display in tag_to_language.items()],
    ["tag", "language"]
)

languages_df.orderBy("language").show(30, truncate=False)
print("Количество языков после нормализации:", languages_df.count())


+------------+------------+
|tag         |language    |
+------------+------------+
|@formula    |@Formula    |
|a#          |A# (Axiom)  |
|a#-.net     |A# .NET     |
|a+          |A+          |
|a++         |A++         |
|a-0-system  |A-0 System  |
|abap        |ABAP        |
|abc         |ABC         |
|abc-algol   |ABC ALGOL   |
|abset       |ABSET       |
|absys       |ABSYS       |
|acc         |ACC         |
|acl2        |ACL2        |
|act-iii     |ACT-III     |
|aimms       |AIMMS       |
|alf         |ALF         |
|algol-58    |ALGOL 58    |
|algol-60    |ALGOL 60    |
|algol-68    |ALGOL 68    |
|algol-w     |ALGOL W     |
|amos        |AMOS        |
|ampl        |AMPL        |
|apl         |APL         |
|arexx       |ARexx       |
|ats         |ATS         |
|awk         |AWK         |
|accent      |Accent      |
|ace-dasl    |Ace DASL    |
|action!     |Action!     |
|actionscript|ActionScript|
+------------+------------+
only showing top 30 rows

Количество языков посл

## 7. Отбор только вопросов за 2010–2020 годы


In [27]:
questions = (
    posts
    .filter(F.col("post_type_id") == 1)          # только вопросы
    .filter(F.col("tags").isNotNull())
    .filter(F.col("creation_date").isNotNull())
    .withColumn("creation_ts", F.to_timestamp("creation_date"))
    .withColumn("year", F.year("creation_ts"))
    .filter((F.col("year") >= 2010) & (F.col("year") <= 2020))
)

questions.select("post_id", "year", "tags").show(20, truncate=False)
print("Количество вопросов за 2010-2020:", questions.count())


+-------+----+---------------------------------------------------------------+
|post_id|year|tags                                                           |
+-------+----+---------------------------------------------------------------+
|3768363|2010|<c++><character-encoding>                                      |
|3775996|2010|<sharepoint><infopath>                                         |
|3776721|2010|<iphone><app-store><in-app-purchase>                           |
|3777993|2010|<symfony1><schema><doctrine><fixtures>                         |
|3778222|2010|<java>                                                         |
|3785460|2010|<visual-studio-2010><stylecop>                                 |
|3789116|2010|<cakephp><file-upload><swfupload>                              |
|3794815|2010|<git><cygwin><putty>                                           |
|3797876|2010|<drupal><drupal-6>                                             |
|3798872|2010|<php><wordpress><memory>              

## 8. Разбор тегов


In [28]:
question_tags = (
    questions
    .withColumn(
        "tag_array",
        F.split(
            F.regexp_replace(
                F.regexp_replace(F.col("tags"), r"^<|>$", ""),
                r"><",
                "|"
            ),
            r"\|"
        )
    )
    .select(
        "post_id",
        "year",
        F.explode("tag_array").alias("tag")
    )
    .withColumn("tag", F.lower(F.col("tag")))
)

question_tags.show(30, truncate=False)


+-------+----+------------------+
|post_id|year|tag               |
+-------+----+------------------+
|3768363|2010|c++               |
|3768363|2010|character-encoding|
|3775996|2010|sharepoint        |
|3775996|2010|infopath          |
|3776721|2010|iphone            |
|3776721|2010|app-store         |
|3776721|2010|in-app-purchase   |
|3777993|2010|symfony1          |
|3777993|2010|schema            |
|3777993|2010|doctrine          |
|3777993|2010|fixtures          |
|3778222|2010|java              |
|3785460|2010|visual-studio-2010|
|3785460|2010|stylecop          |
|3789116|2010|cakephp           |
|3789116|2010|file-upload       |
|3789116|2010|swfupload         |
|3794815|2010|git               |
|3794815|2010|cygwin            |
|3794815|2010|putty             |
|3797876|2010|drupal            |
|3797876|2010|drupal-6          |
|3798872|2010|php               |
|3798872|2010|wordpress         |
|3798872|2010|memory            |
|3801290|2010|c#                |
|3801290|2010|

## 9. Оставляем только теги, которые являются языками программирования


In [29]:
language_questions = (
    question_tags
    .join(F.broadcast(languages_df), on="tag", how="inner")
)

language_questions.show(30, truncate=False)
print("Количество совпадений тегов с языками:", language_questions.count())


+-----------+-------+----+-------------------+
|tag        |post_id|year|language           |
+-----------+-------+----+-------------------+
|c++        |3768363|2010|C++ – ISO/IEC 14882|
|java       |3778222|2010|Java               |
|php        |3798872|2010|PHP                |
|c#         |3801290|2010|C# – ISO/IEC 23270 |
|c#         |3802658|2010|C# – ISO/IEC 23270 |
|ruby       |3833655|2010|Ruby               |
|c          |3838866|2010|C                  |
|c#         |3841866|2010|C# – ISO/IEC 23270 |
|c#         |3858524|2010|C# – ISO/IEC 23270 |
|php        |3859099|2010|PHP                |
|python     |3872977|2010|Python             |
|javascript |3885802|2010|JavaScript         |
|applescript|3886770|2010|AppleScript        |
|c#         |3890854|2010|C# – ISO/IEC 23270 |
|php        |3891676|2010|PHP                |
|c++        |3901256|2010|C++ – ISO/IEC 14882|
|php        |3904423|2010|PHP                |
|c#         |3914572|2010|C# – ISO/IEC 23270 |
|javascript |

## 10. Подсчёт количества вопросов по паре (год, язык)


In [30]:
yearly_counts = (
    language_questions
    .groupBy("year", "language")
    .agg(F.countDistinct("post_id").alias("question_count"))
)

yearly_counts.orderBy("year", F.desc("question_count"), "language").show(50, truncate=False)


+----+-------------------------------------------------+--------------+
|year|language                                         |question_count|
+----+-------------------------------------------------+--------------+
|2010|C# – ISO/IEC 23270                               |96            |
|2010|Java                                             |52            |
|2010|PHP                                              |46            |
|2010|JavaScript                                       |44            |
|2010|C++ – ISO/IEC 14882                              |28            |
|2010|Python                                           |26            |
|2010|Objective-C                                      |23            |
|2010|C                                                |20            |
|2010|Ruby                                             |12            |
|2010|Delphi                                           |8             |
|2010|AppleScript                                      |3       

## 11. Построение top-10 языков для каждого года


In [31]:
w = Window.partitionBy("year").orderBy(
    F.desc("question_count"),
    F.asc("language")
)

result = (
    yearly_counts
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= 10)
    .select("year", "rank", "language", "question_count")
    .orderBy("year", "rank")
)

result.show(200, truncate=False)


+----+----+-----------------------------------+--------------+
|year|rank|language                           |question_count|
+----+----+-----------------------------------+--------------+
|2010|1   |C# – ISO/IEC 23270                 |96            |
|2010|2   |Java                               |52            |
|2010|3   |PHP                                |46            |
|2010|4   |JavaScript                         |44            |
|2010|5   |C++ – ISO/IEC 14882                |28            |
|2010|6   |Python                             |26            |
|2010|7   |Objective-C                        |23            |
|2010|8   |C                                  |20            |
|2010|9   |Ruby                               |12            |
|2010|10  |Delphi                             |8             |
|2011|1   |PHP                                |102           |
|2011|2   |C# – ISO/IEC 23270                 |100           |
|2011|3   |Java                               |93      

## 12. Сохранение результата в Apache Parquet


In [32]:
(
    result
    .write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

print("Результат сохранён в:", OUTPUT_PATH)


Результат сохранён в: /content/drive/MyDrive/BD_LR/top_languages_2010_2020.parquet


## 13. Проверка сохранённого Parquet


In [33]:
saved_result = spark.read.parquet(OUTPUT_PATH)
saved_result.orderBy("year", "rank").show(200, truncate=False)


+----+----+-----------------------------------+--------------+
|year|rank|language                           |question_count|
+----+----+-----------------------------------+--------------+
|2010|1   |C# – ISO/IEC 23270                 |96            |
|2010|2   |Java                               |52            |
|2010|3   |PHP                                |46            |
|2010|4   |JavaScript                         |44            |
|2010|5   |C++ – ISO/IEC 14882                |28            |
|2010|6   |Python                             |26            |
|2010|7   |Objective-C                        |23            |
|2010|8   |C                                  |20            |
|2010|9   |Ruby                               |12            |
|2010|10  |Delphi                             |8             |
|2011|1   |PHP                                |102           |
|2011|2   |C# – ISO/IEC 23270                 |100           |
|2011|3   |Java                               |93      

## 14. Завершение работы


In [34]:
spark.stop()
